# Week 3 Homework

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [3]:
predict_conversion_df = pd.read_csv("digital_marketing_campaign_dataset.csv")
google_ads_df = pd.read_csv("GoogleAds_DataAnalytics_Sales_Uncleaned.csv")
marketing_products_df = pd.read_csv("marketing_and_product_performance.csv")

## Backward Selection for Regression of predict_conversion_df

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector

# Data Preparation
df = predict_conversion_df.drop(columns=['CustomerID'])

# Convert categorical features into dummy/indicator variables
X = pd.get_dummies(df.drop(columns=['Conversion']), drop_first=True)
y = df['Conversion']

# Set up Linear Regression and Backward Selection
model = LinearRegression()
# n_features_to_select='auto' will select half of the features by default.
sfs = SequentialFeatureSelector(model, direction='backward', scoring='r2', cv=5)

# Fit the Backward Selection model
print("Running backward selection (this might take a moment)...")
sfs.fit(X, y)

# Display Results
selected_features = X.columns[sfs.get_support()]
print(f"Original number of features: {X.shape[1]}")
print(f"Selected number of features: {len(selected_features)}\n")

print("Selected features:")
for feature in selected_features:
    print(f"- {feature}")

# Fit final model with the selected features
model.fit(X[selected_features], y)
print(f"\nFinal Model R-squared: {model.score(X[selected_features], y):.4f}")

Running backward selection (this might take a moment)...
Original number of features: 21
Selected number of features: 11

Selected features:
- AdSpend
- ClickThroughRate
- ConversionRate
- WebsiteVisits
- PagesPerVisit
- TimeOnSite
- EmailOpens
- EmailClicks
- PreviousPurchases
- LoyaltyPoints
- CampaignType_Conversion

Final Model R-squared: 0.1387


## PLSR of predict_conversion_df

In [6]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict
import numpy as np

# Scaling the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Iterate through components to find optimal number using 5-fold CV
components = np.arange(1, X.shape[1] + 1)
cv_r2_scores = []

for i in components:
    pls = PLSRegression(n_components=i)
    y_cv = cross_val_predict(pls, X_scaled, y, cv=5)
    r2 = r2_score(y, y_cv)
    cv_r2_scores.append(r2)

optimal_components = components[np.argmax(cv_r2_scores)]
best_cv_r2 = max(cv_r2_scores)

print(f"Optimal number of PLS components: {optimal_components}")
print(f"Best Cross-Validated R-squared: {best_cv_r2:.4f}\n")

# Fit the final PLS model with the optimal number of components
pls_final = PLSRegression(n_components=optimal_components)
pls_final.fit(X_scaled, y)

print(f"Training R-squared (with {optimal_components} components): {pls_final.score(X_scaled, y):.4f}")

Optimal number of PLS components: 4
Best Cross-Validated R-squared: 0.1044

Training R-squared (with 4 components): 0.1396


## Backward Selection for Regression of google_ads_df

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Data Preparation
df_ads = google_ads_df.copy()

# Convert 'Conversion Rate' to numeric if it's strings
if df_ads['Conversion Rate'].dtype == object:
    df_ads['Conversion Rate'] = df_ads['Conversion Rate'].str.rstrip('%').astype('float') / 100.0

# Drop identifiers and textual columns
# Also drop 'Conversion Rate' and 'Sale_Amount' to prevent target leakage when predicting 'Conversions'
cols_to_drop = ['Ad_ID', 'Campaign_Name', 'Ad_Date', 'Keyword', 'Conversion Rate', 'Sale_Amount']
df_ads = df_ads.drop(columns=[c for c in cols_to_drop if c in df_ads.columns])
df_ads = df_ads.dropna()

# Categorical variables to dummies
X_ads = pd.get_dummies(df_ads.drop(columns=['Conversions']), drop_first=True)
y_ads = df_ads['Conversions']

# Scale features to avoid SVD convergence issues
scaler = StandardScaler()
X_ads_scaled = pd.DataFrame(scaler.fit_transform(X_ads), columns=X_ads.columns)

# 2. Linear Regression Model
model_ads = LinearRegression()

# 3. Apply Recursive Feature Elimination (RFE)
# RFE starts with all features and drops the least important one by one (Backward Selection)
# We will select the top 50% of features (step=1 eliminates one feature per step)
rfe = RFE(estimator=model_ads, n_features_to_select=X_ads.shape[1] // 2, step=1)

print("Running fast backward selection (RFE) on google_ads_df...")
rfe.fit(X_ads_scaled, y_ads)

# 4. Display Results
selected_features_ads = X_ads.columns[rfe.support_]
print(f"Original number of features: {X_ads.shape[1]}")
print(f"Selected number of features: {len(selected_features_ads)}\n")

print("Selected features:")
for feature in selected_features_ads:
    print(f"- {feature}")

# Evaluate the model with selected features
model_ads.fit(X_ads_scaled[selected_features_ads], y_ads)
r2_score = model_ads.score(X_ads_scaled[selected_features_ads], y_ads)
print(f"\nFinal Model R-squared: {r2_score:.4f}")

Running fast backward selection (RFE) on google_ads_df...


KeyboardInterrupt: 